In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("credit_risk_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df.head()

Shape: (32581, 12)

Columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']

Missing values:
 person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Duplicate rows: 165


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [2]:
import sqlite3

conn = sqlite3.connect("loans.db")
df.to_sql("loans", conn, if_exists="replace", index=False)

def run(query):
    return pd.read_sql_query(query, conn)

q1 = """
SELECT COUNT(*) AS total_loans,
       ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loans
"""
print("Q1 - Overall\n", run(q1))

q2 = """
SELECT loan_grade,
       COUNT(*) AS loans,
       ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct,
       ROUND(AVG(loan_int_rate), 2) AS avg_interest_rate
FROM loans
GROUP BY loan_grade
ORDER BY loan_grade
"""
print("\nQ2 - By loan grade\n", run(q2))

q3 = """
SELECT person_home_ownership,
       COUNT(*) AS loans,
       ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY person_home_ownership
ORDER BY default_rate_pct DESC
"""
print("\nQ3 - By home ownership\n", run(q3))

q4 = """
SELECT MIN(person_age) AS min_age,
       MAX(person_age) AS max_age,
       MAX(person_emp_length) AS max_emp_length,
       MAX(person_income) AS max_income,
       SUM(CASE WHEN person_age > 100 THEN 1 ELSE 0 END) AS age_over_100,
       SUM(CASE WHEN person_emp_length > 60 THEN 1 ELSE 0 END) AS emp_over_60
FROM loans
"""
print("\nQ4 - Outlier check\n", run(q4))

Q1 - Overall
    total_loans  default_rate_pct
0        32581             21.82

Q2 - By loan grade
   loan_grade  loans  default_rate_pct  avg_interest_rate
0          A  10777              9.96               7.33
1          B  10451             16.28              11.00
2          C   6458             20.73              13.46
3          D   3626             59.05              15.36
4          E    964             64.42              17.01
5          F    241             70.54              18.61
6          G     64             98.44              20.25

Q3 - By home ownership
   person_home_ownership  loans  default_rate_pct
0                  RENT  16446             31.57
1                 OTHER    107             30.84
2              MORTGAGE  13444             12.57
3                   OWN   2584              7.47

Q4 - Outlier check
    min_age  max_age  max_emp_length  max_income  age_over_100  emp_over_60
0       20      144           123.0     6000000             5            2


In [3]:
df_clean = df.copy()
print("Rows before cleaning:", len(df_clean))

df_clean = df_clean.drop_duplicates()
print("After removing duplicates:", len(df_clean))

df_clean = df_clean[df_clean["person_age"] <= 100]
print("After removing age > 100:", len(df_clean))

df_clean = df_clean[(df_clean["person_emp_length"] <= 60) | (df_clean["person_emp_length"].isnull())]
print("After removing emp_length > 60:", len(df_clean))

df_clean["loan_int_rate"] = df_clean["loan_int_rate"].fillna(
    df_clean.groupby("loan_grade")["loan_int_rate"].transform("median")
)

df_clean["person_emp_length"] = df_clean["person_emp_length"].fillna(
    df_clean["person_emp_length"].median()
)

print("\nMissing values after cleaning:\n", df_clean.isnull().sum())
print("\nFinal shape:", df_clean.shape)
print("Default rate:", round(df_clean["loan_status"].mean() * 100, 2), "%")

df_clean.describe().round(2)

Rows before cleaning: 32581
After removing duplicates: 32416
After removing age > 100: 32411
After removing emp_length > 60: 32409

Missing values after cleaning:
 person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

Final shape: (32409, 12)
Default rate: 21.87 %


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32409.00,32409.00,32409.00,32409.00,32409.00,32409.00,32409.00,32409.00
mean,27.73,65894.28,4.76,9592.49,11.02,0.22,0.17,5.81
std,6.21,52517.87,3.98,6320.89,3.21,0.41,0.11,4.06
min,20.00,4000.00,0.00,500.00,5.42,0.00,0.00,2.00
25%,23.00,38500.00,2.00,5000.00,7.88,0.00,0.09,3.00
50%,26.00,55000.00,4.00,8000.00,10.99,0.00,0.15,4.00
75%,30.00,79200.00,7.00,12250.00,13.48,0.00,0.23,8.00
max,94.00,2039784.00,41.00,35000.00,23.22,1.00,0.83,30.00


In [4]:
import plotly.express as px

sample = df_clean.sample(5000, random_state=42).copy()

income_cap = df_clean["person_income"].quantile(0.99)
sample = sample[sample["person_income"] <= income_cap]

sample["Status"] = sample["loan_status"].map({0: "Repaid", 1: "Defaulted"})

fig = px.scatter_3d(
    sample,
    x="person_age",
    y="person_income",
    z="loan_amnt",
    color="Status",
    color_discrete_map={"Repaid": "#00e5ff", "Defaulted": "#ff1744"},
    opacity=0.65,
    title="Loan Default Landscape: Age vs Income vs Loan Amount",
    labels={
        "person_age": "Age",
        "person_income": "Annual Income",
        "loan_amnt": "Loan Amount",
    },
)

fig.update_traces(marker=dict(size=2.5))

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    legend_title_text="",
    height=700,
    scene=dict(
        bgcolor="#0d0d0d",
        xaxis=dict(backgroundcolor="#0d0d0d", gridcolor="#2a2a2a"),
        yaxis=dict(backgroundcolor="#0d0d0d", gridcolor="#2a2a2a"),
        zaxis=dict(backgroundcolor="#0d0d0d", gridcolor="#2a2a2a"),
    ),
)

fig.show()

In [5]:
import plotly.graph_objects as go

eda = df_clean.copy()

eda["income_band"] = pd.qcut(
    eda["person_income"], 5,
    labels=["Lowest 20%", "Lower-mid", "Middle", "Upper-mid", "Highest 20%"]
)

eda["lti_band"] = pd.cut(
    eda["loan_percent_income"],
    bins=[-0.01, 0.10, 0.20, 0.30, 1.0],
    labels=["0-10%", "10-20%", "20-30%", "30%+"]
)

rate = eda.pivot_table(index="income_band", columns="lti_band",
                       values="loan_status", aggfunc="mean", observed=True) * 100
count = eda.pivot_table(index="income_band", columns="lti_band",
                        values="loan_status", aggfunc="count", observed=True)

text = [[f"{rate.iloc[i, j]:.1f}%<br>n={count.iloc[i, j]}"
         for j in range(rate.shape[1])] for i in range(rate.shape[0])]

fig = go.Figure(go.Heatmap(
    z=rate.values,
    x=list(rate.columns),
    y=list(rate.index),
    text=text,
    texttemplate="%{text}",
    colorscale=[[0, "#16213e"], [0.5, "#c2185b"], [1, "#ff1744"]],
    colorbar=dict(title="Default %"),
))

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    plot_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    title="Default Rate: Income Band vs Loan-to-Income Ratio",
    xaxis_title="Loan as % of Income",
    yaxis_title="Income Band",
    height=550,
)

fig.show()

In [6]:
import plotly.graph_objects as go

g = (df_clean.groupby(["loan_intent", "cb_person_default_on_file"])["loan_status"]
     .agg(rate="mean", n="count").reset_index())
g["rate"] = g["rate"] * 100

rate = g.pivot(index="loan_intent", columns="cb_person_default_on_file", values="rate")
cnt = g.pivot(index="loan_intent", columns="cb_person_default_on_file", values="n")

order = rate["N"].sort_values().index
rate = rate.loc[order]
cnt = cnt.loc[order]

fig = go.Figure()

for intent in rate.index:
    fig.add_trace(go.Scatter(
        x=[rate.loc[intent, "N"], rate.loc[intent, "Y"]],
        y=[intent, intent],
        mode="lines",
        line=dict(color="#555555", width=3),
        showlegend=False,
        hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(
    x=rate["N"], y=rate.index, mode="markers+text", name="No prior default",
    marker=dict(color="#00e5ff", size=16),
    text=[f"{v:.1f}%" for v in rate["N"]], textposition="top center",
    customdata=cnt["N"], hovertemplate="%{y}<br>%{x:.1f}% (n=%{customdata})<extra></extra>",
))

# Red dots: prior default on file
fig.add_trace(go.Scatter(
    x=rate["Y"], y=rate.index, mode="markers+text", name="Prior default on file",
    marker=dict(color="#ff1744", size=16),
    text=[f"{v:.1f}%" for v in rate["Y"]], textposition="top center",
    customdata=cnt["Y"], hovertemplate="%{y}<br>%{x:.1f}% (n=%{customdata})<extra></extra>",
))

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    plot_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    title="Does a Past Default Matter More Than Loan Purpose?",
    xaxis_title="Default Rate (%)",
    xaxis=dict(gridcolor="#2a2a2a", range=[0, 60]),
    yaxis=dict(gridcolor="#2a2a2a"),
    height=550,
)

fig.show()

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

model_df = pd.get_dummies(
    df_clean,
    columns=["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"],
    drop_first=True,
    dtype=int,
)

y = model_df["loan_status"]
X_all = model_df.drop(columns=["loan_status"])

lender_cols = [c for c in X_all.columns if c.startswith("loan_grade_")] + ["loan_int_rate"]
X_no_lender = X_all.drop(columns=lender_cols)

feature_sets = {"With grade + rate": X_all, "Without grade + rate": X_no_lender}

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced", min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
}

results = []
for fs_name, X in feature_sets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    for m_name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        results.append({
            "Features": fs_name,
            "Model": m_name,
            "Recall": recall_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "F1": f1_score(y_test, pred),
            "ROC-AUC": roc_auc_score(y_test, proba),
        })

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
pd.DataFrame(results).round(3)

Train rows: 25927 | Test rows: 6482


,Features,Model,Recall,Precision,F1,ROC-AUC
0,With grade + rate,Logistic Regression,0.781,0.538,0.637,0.871
1,With grade + rate,Random Forest,0.766,0.845,0.804,0.933
2,Without grade + rate,Logistic Regression,0.704,0.442,0.543,0.807
3,Without grade + rate,Random Forest,0.626,0.736,0.677,0.878


In [8]:
from sklearn.metrics import roc_curve
from sklearn.base import clone
import plotly.graph_objects as go

fitted = {}
probas = {}
for fs_name, X in feature_sets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    for m_name, model in models.items():
        m = clone(model)
        m.fit(X_train, y_train)
        fitted[(fs_name, m_name)] = m
        probas[(fs_name, m_name)] = m.predict_proba(X_test)[:, 1]

styles = {
    ("With grade + rate", "Random Forest"):         dict(color="#ff1744", dash="solid"),
    ("Without grade + rate", "Random Forest"):      dict(color="#ff1744", dash="dash"),
    ("With grade + rate", "Logistic Regression"):   dict(color="#00e5ff", dash="solid"),
    ("Without grade + rate", "Logistic Regression"):dict(color="#00e5ff", dash="dash"),
}

fig = go.Figure()

for key, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    auc = roc_auc_score(y_test, p)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{key[1]} | {key[0]} (AUC {auc:.3f})",
        line=dict(width=3, **styles[key]),
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random guess",
    line=dict(color="#666666", width=2, dash="dot"),
))

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    plot_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    title="ROC Curves: What the Lender's Own Risk Columns Are Worth",
    xaxis=dict(title="False Positive Rate", gridcolor="#2a2a2a", range=[0, 1]),
      yaxis=dict(title="True Positive Rate (Recall)", gridcolor="#2a2a2a", range=[0, 1.02]),
    legend=dict(x=0.45, y=0.05, bgcolor="rgba(13,13,13,0.8)"),
    height=700,
)

fig.show()

In [9]:
from sklearn.metrics import precision_recall_curve

key = ("Without grade + rate", "Random Forest")
p = probas[key]

prec, rec, thr = precision_recall_curve(y_test, p)

prec, rec = prec[:-1], rec[:-1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=thr, y=rec, mode="lines", name="Recall (defaults caught)",
                         line=dict(color="#ff1744", width=3)))
fig.add_trace(go.Scatter(x=thr, y=prec, mode="lines", name="Precision (flags that were right)",
                         line=dict(color="#00e5ff", width=3)))
fig.add_vline(x=0.5, line=dict(color="#888888", dash="dot"),
              annotation_text="default 0.5", annotation_font_color="#e0e0e0")

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    plot_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    title="The Threshold Trade-off: Catching Defaults vs False Alarms",
    xaxis=dict(title="Decision Threshold (flag as default if probability ≥ this)",
               gridcolor="#2a2a2a", range=[0, 1]),
    yaxis=dict(title="Score", gridcolor="#2a2a2a", range=[0, 1.02]),
    legend=dict(x=0.02, y=0.05, bgcolor="rgba(13,13,13,0.8)"),
    height=600,
)
fig.show()

# Numbers at specific thresholds
print("Primary model: Random Forest without grade + rate\n")
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred = (p >= t).astype(int)
    print(f"threshold {t:.2f} | recall {recall_score(y_test, pred):.3f} | "
          f"precision {precision_score(y_test, pred):.3f} | "
          f"flagged {pred.mean() * 100:.1f}% of loans")

Primary model: Random Forest without grade + rate

threshold 0.20 | recall 0.913 | precision 0.350 | flagged 57.1% of loans
threshold 0.25 | recall 0.865 | precision 0.407 | flagged 46.5% of loans
threshold 0.30 | recall 0.816 | precision 0.470 | flagged 38.0% of loans
threshold 0.35 | recall 0.770 | precision 0.538 | flagged 31.3% of loans
threshold 0.40 | recall 0.722 | precision 0.607 | flagged 26.0% of loans
threshold 0.50 | recall 0.626 | precision 0.736 | flagged 18.6% of loans


In [10]:
from sklearn.metrics import confusion_matrix

THRESHOLD = 0.35
key = ("Without grade + rate", "Random Forest")

pred = (probas[key] >= THRESHOLD).astype(int)

cm = confusion_matrix(y_test, pred)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

tn, fp, fn, tp = cm.ravel()

labels_x = ["Predicted Repaid", "Predicted Default"]
labels_y = ["Actually Repaid", "Actually Defaulted"]

text = [[f"{cm[i, j]:,}<br>{cm_pct[i, j]:.1f}% of row" for j in range(2)]
        for i in range(2)]

fig = go.Figure(go.Heatmap(
    z=cm_pct, x=labels_x, y=labels_y,
    text=text, texttemplate="%{text}", textfont=dict(size=18, color="#ffffff"),
    colorscale=[[0, "#16213e"], [1, "#0097a7"]],
    zmin=0, zmax=100, showscale=False,
))

fig.update_yaxes(autorange="reversed")

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    plot_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    title=f"Confusion Matrix at Threshold {THRESHOLD} (Random Forest, borrower data only)",
    height=520,
)

fig.show()

print(f"Correctly approved (true negatives):  {tn:,}")
print(f"Good borrowers flagged (false positives): {fp:,}")
print(f"Defaults missed (false negatives):    {fn:,}")
print(f"Defaults caught (true positives):     {tp:,}")

Correctly approved (true negatives):  4,125
Good borrowers flagged (false positives): 939
Defaults missed (false negatives):    326
Defaults caught (true positives):     1,092


In [11]:
key = ("Without grade + rate", "Random Forest")
rf = fitted[key]

imp = (pd.Series(rf.feature_importances_, index=X_no_lender.columns)
       .sort_values(ascending=True))
imp = imp.tail(12)

fig = go.Figure(go.Bar(
    x=imp.values, y=imp.index, orientation="h",
    marker=dict(color=imp.values, colorscale=[[0, "#16213e"], [1, "#ff1744"]]),
    text=[f"{v:.3f}" for v in imp.values], textposition="outside",
))

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d0d0d",
    plot_bgcolor="#0d0d0d",
    font=dict(color="#e0e0e0"),
    title="What Drives Default Risk? (Random Forest, borrower data only)",
xaxis=dict(title="Importance", gridcolor="#2a2a2a", range=[0, 0.3]),    height=600,
)
fig.show()

In [12]:
import joblib
import os
from google.colab import files

key = ("Without grade + rate", "Random Forest")
rf = fitted[key]

meta = {
    "columns": list(X_no_lender.columns),
    "threshold": 0.35,
    "home_options": sorted(df_clean["person_home_ownership"].unique()),
    "intent_options": sorted(df_clean["loan_intent"].unique()),
    "median_cred_hist": float(df_clean["cb_person_cred_hist_length"].median()),
    "median_emp_length": float(df_clean["person_emp_length"].median()),
}

joblib.dump(rf, "risk_model.joblib", compress=3)
joblib.dump(meta, "model_meta.joblib")

for f in ["risk_model.joblib", "model_meta.joblib"]:
    print(f, round(os.path.getsize(f) / 1_000_000, 1), "MB")

loaded = joblib.load("risk_model.joblib")
print("Saved model probability:", loaded.predict_proba(X_no_lender.iloc[[0]])[0, 1])
print("Original model probability:", rf.predict_proba(X_no_lender.iloc[[0]])[0, 1])

files.download("risk_model.joblib")
files.download("model_meta.joblib")

risk_model.joblib 14.6 MB
model_meta.joblib 0.0 MB
Saved model probability: 0.3340749377985041
Original model probability: 0.33407493779850417


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Step 7B - Record library versions for the app
import sklearn, pandas, numpy, joblib

print("scikit-learn", sklearn.__version__)
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("joblib", joblib.__version__)

scikit-learn 1.6.1
pandas 2.2.3
numpy 2.1.3
joblib 1.6.0
